# Akkadian LoRA Fine-tuning + Submission

ByT5 blended model + LoRA fine-tuning on Akkademia corpus.

**Required Kaggle Datasets:**
- `deep-past-initiative-machine-translation` (competition data)
- `byt5-base-big-data2` (model 1)
- `byt5-akkadian-model` (model 2)
- `train-gap-all-2` (model 3)

**Required packages:** `pip install peft`

In [ ]:
!pip install -q peft

In [ ]:
import re
import time
import math
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    Seq2SeqTrainer, Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq,
)
from peft import LoraConfig, get_peft_model, TaskType

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## 1. Configuration

In [ ]:
CFG = {
    # Kaggle paths
    "competition_data": "/kaggle/input/deep-past-initiative-machine-translation",
    "models": [
        "/kaggle/input/byt5-base-big-data2",
        "/kaggle/input/byt5-akkadian-model",
        "/kaggle/input/train-gap-all-2/byt5-base-akkadian_gap_setence2",
    ],
    "weights": [0.99, 0.98, 0.39],
    
    # LoRA
    "lora_rank": 8,
    "lora_alpha": 16,
    "lora_dropout": 0.05,
    "target_modules": ["q", "v", "o"],
    
    # Training
    "epochs": 3,
    "lr": 3e-4,
    "batch_size": 4,
    "grad_accum": 4,
    "max_source_len": 512,
    "max_target_len": 256,
    "warmup_ratio": 0.05,
    "comp_weight": 3,  # Competition data repeated 3x
    
    # Generation
    "num_beams": 12,
    "num_return_sequences": 1,
    "max_new_tokens": 496,
    "length_penalty": 1.3,
}

## 2. Download Akkademia Data

In [ ]:
import urllib.request
import os

AKK_DIR = "/kaggle/working/akkademia"
os.makedirs(AKK_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/gaigutherz/Akkademia/master/NMT_input"
for fname in ["train.tr", "train.en", "valid.tr", "valid.en"]:
    path = os.path.join(AKK_DIR, fname)
    if not os.path.exists(path):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}", path)
    else:
        print(f"{fname} already exists")

with open(os.path.join(AKK_DIR, "train.tr")) as f:
    print(f"Akkademia train: {sum(1 for _ in f)} lines")

## 3. Data Preparation

In [ ]:
def normalize_akkademia_transliteration(text):
    """Convert Akkademia format to competition format.
    {d}-X -> (d)X, {X} -> X
    """
    text = re.sub(r"\{([^}]+)\}-", r"(\1)", text)
    text = re.sub(r"\{([^}]+)\}", r"\1", text)
    return text


def normalize_akkademia_translation(text):
    """Clean Akkademia English translations."""
    text = re.sub(r'\(lit\.\s*"[^"]*"\)', '', text)
    text = re.sub(r'\(DN\s+and\)', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Load Akkademia data
with open(os.path.join(AKK_DIR, "train.tr")) as f:
    akk_src = [normalize_akkademia_transliteration(l.strip()) for l in f]
with open(os.path.join(AKK_DIR, "train.en")) as f:
    akk_tgt = [normalize_akkademia_translation(l.strip()) for l in f]

# Filter empty/short pairs
paired = [(s, t) for s, t in zip(akk_src, akk_tgt)
          if s.strip() and t.strip() and len(s) > 5 and len(t) > 5]
akk_src, akk_tgt = zip(*paired) if paired else ([], [])

# Load competition data
comp_df = pd.read_csv(os.path.join(CFG["competition_data"], "train.csv"))
comp_paired = [(str(s), str(t)) for s, t in zip(comp_df["transliteration"], comp_df["translation"])
               if isinstance(s, str) and isinstance(t, str) and s.strip() and t.strip()]
comp_src, comp_tgt = zip(*comp_paired) if comp_paired else ([], [])

# Combine with competition data weighted 3x
all_src = list(akk_src) + list(comp_src) * CFG["comp_weight"]
all_tgt = list(akk_tgt) + list(comp_tgt) * CFG["comp_weight"]

print(f"Akkademia: {len(akk_src)} pairs")
print(f"Competition: {len(comp_src)} pairs (x{CFG['comp_weight']} = {len(comp_src)*CFG['comp_weight']})")
print(f"Total training: {len(all_src)} pairs")

## 4. Model Loading + Blending

In [ ]:
def load_blended_model():
    """Load 3 ByT5 models and blend weights."""
    total = sum(CFG["weights"])
    W = [w / total for w in CFG["weights"]]
    
    print("Loading model 1...")
    sd_m1 = AutoModelForSeq2SeqLM.from_pretrained(CFG["models"][0]).state_dict()
    print("Loading model 2 (base)...")
    base_model = AutoModelForSeq2SeqLM.from_pretrained(CFG["models"][1])
    final_sd = base_model.state_dict()
    print("Loading model 3...")
    sd_m3 = AutoModelForSeq2SeqLM.from_pretrained(CFG["models"][2]).state_dict()
    
    print("Blending weights...")
    for k in final_sd:
        val = W[1] * final_sd[k]
        norm = W[1]
        if k in sd_m1:
            val = val + W[0] * sd_m1[k]
            norm = norm + W[0]
        if k in sd_m3:
            val = val + W[2] * sd_m3[k]
            norm = norm + W[2]
        final_sd[k] = val / norm
    
    base_model.load_state_dict(final_sd)
    model = base_model.float()
    del sd_m1, sd_m3
    torch.cuda.empty_cache()
    
    tokenizer = AutoTokenizer.from_pretrained(CFG["models"][1])
    param_count = sum(p.numel() for p in model.parameters())
    print(f"Blended model: {param_count:,} parameters")
    return model, tokenizer

model, tokenizer = load_blended_model()

## 5. Apply LoRA

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=CFG["lora_rank"],
    lora_alpha=CFG["lora_alpha"],
    lora_dropout=CFG["lora_dropout"],
    target_modules=CFG["target_modules"],
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 6. Dataset for Trainer

In [ ]:
from datasets import Dataset as HFDataset

TASK_PREFIX = "translate Akkadian to English: "

def preprocess_function(examples):
    inputs = [TASK_PREFIX + s for s in examples["source"]]
    targets = examples["target"]
    
    model_inputs = tokenizer(
        inputs, max_length=CFG["max_source_len"],
        truncation=True, padding=False,
    )
    labels = tokenizer(
        targets, max_length=CFG["max_target_len"],
        truncation=True, padding=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Create HuggingFace dataset
train_ds = HFDataset.from_dict({"source": all_src, "target": all_tgt})
train_ds = train_ds.map(preprocess_function, batched=True, remove_columns=["source", "target"])

print(f"Train dataset: {len(train_ds)} samples")
print(f"Sample input length: {len(train_ds[0]['input_ids'])} tokens")

## 7. Training with HuggingFace Trainer

In [ ]:
data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/lora_checkpoints",
    num_train_epochs=CFG["epochs"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    learning_rate=CFG["lr"],
    weight_decay=0.01,
    warmup_ratio=CFG["warmup_ratio"],
    lr_scheduler_type="cosine",
    fp16=True,
    logging_steps=50,
    save_strategy="epoch",
    save_total_limit=2,
    report_to="none",
    dataloader_num_workers=2,
    max_grad_norm=1.0,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    data_collator=data_collator,
    tokenizer=tokenizer,
)

print("Starting training...")
start_time = time.time()
trainer.train()
elapsed = (time.time() - start_time) / 60
print(f"Training complete in {elapsed:.1f} minutes")

## 8. Save LoRA Adapter

In [ ]:
# Save the LoRA adapter
adapter_path = "/kaggle/working/lora_adapter"
model.save_pretrained(adapter_path)
print(f"LoRA adapter saved to {adapter_path}")

# Merge LoRA into base model for inference
model = model.merge_and_unload()
model.eval()
print("LoRA merged into base model")

## 9. Post-processing Functions

In [ ]:
_SUBSCRIPT_TABLE = str.maketrans("\u2080\u2081\u2082\u2083\u2084\u2085\u2086\u2087\u2088\u2089", "0123456789")
_SUPERSCRIPT_TABLE = str.maketrans("\u2070\u00b9\u00b2\u00b3\u2074\u2075\u2076\u2077\u2078\u2079", "0123456789")
_BAD_OUTPUT_CHARS = '!?()\"\u2014\u2013<>\u2308\u230b\u230a[]+\u02be/;'

_SHORT_INPUT_MAP = {
    "a-na": "To", "um-ma": "saying:", "IGI": "Witnesses:",
    "KIŠIB": "Seal of", "ma-na": "mina of silver",
    "šu-ma": "If he does not pay", "i-na": "In",
    "ša": "of", "u": "and", "ki-ma": "like",
    "DUMU": "son of", "ITI": "Month:", "MU": "Year:",
    "LUGAL": "king", "šar-ri": "king", "DINGIR": "god",
    "KUR": "land", "URU": "city", "NAM": "fate",
    "A": "water", "É": "house", "GIŠ": "wood",
}


def preprocess_input(text):
    if pd.isna(text): return ""
    t = str(text)
    t = re.sub(r'(\.{3,}|\u2026+|\u2026\u2026)', '<big_gap>', t)
    t = re.sub(r'(xx+|\s+x\s+)', '<gap>', t)
    t = t.translate(_SUBSCRIPT_TABLE).translate(_SUPERSCRIPT_TABLE)
    t = re.sub(r'\s+', ' ', t).strip()
    return TASK_PREFIX + t


def postprocess_output(text, raw_input):
    if not isinstance(text, str): return ""
    t = text.translate(str.maketrans('', '', _BAD_OUTPUT_CHARS))
    t = t.replace('<big_gap>', '').replace('<gap>', '')
    t = re.sub(r'\s+', ' ', t).strip().strip('-')
    # Cap output length
    input_len = len(str(raw_input))
    max_len = int(input_len * 0.5 + 30)
    if len(t) > max_len:
        words = t.split()
        result = []
        length = 0
        for w in words:
            if length + len(w) + 1 > max_len and length > 0:
                break
            result.append(w)
            length += len(w) + 1
        t = " ".join(result)
    return t


def get_short_translation(raw_input):
    tokens = str(raw_input).strip().split()
    if len(tokens) == 1:
        return _SHORT_INPUT_MAP.get(tokens[0])
    return None


def is_broken_text(raw_input):
    t = str(raw_input).strip()
    if t.startswith("[") and "..." in t[:20]: return True
    tokens = t.split()
    if len(tokens) > 3:
        gap_count = sum(1 for tok in tokens if tok in ("x", "xx", "xxx", "<gap>", "<big_gap>", "..."))
        if gap_count / len(tokens) > 0.4: return True
    return False

## 10. Inference + Submission

In [ ]:
# Load test data
test_df = pd.read_csv(os.path.join(CFG["competition_data"], "test.csv"))
print(f"Test samples: {len(test_df)}")

results = []
model = model.to(DEVICE)

with torch.inference_mode():
    for idx, row in test_df.iterrows():
        raw_text = str(row["transliteration"]) if pd.notna(row["transliteration"]) else ""
        
        # Short input lookup
        short = get_short_translation(raw_text)
        if short is not None:
            results.append({"id": row["id"], "translation": short})
            continue
        
        # Broken text
        if is_broken_text(raw_text):
            results.append({"id": row["id"], "translation": "..."})
            continue
        
        # Model inference
        input_text = preprocess_input(raw_text)
        inputs = tokenizer(
            input_text, max_length=496,
            padding=True, truncation=True, return_tensors="pt"
        ).to(DEVICE)
        
        outputs = model.generate(
            input_ids=inputs.input_ids,
            attention_mask=inputs.attention_mask,
            num_beams=CFG["num_beams"],
            num_return_sequences=CFG["num_return_sequences"],
            max_new_tokens=CFG["max_new_tokens"],
            length_penalty=CFG["length_penalty"],
            early_stopping=True,
        )
        
        decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
        cleaned = postprocess_output(decoded, raw_text)
        results.append({"id": row["id"], "translation": cleaned})
        print(f"[{idx+1}/{len(test_df)}] {raw_text[:50]}... -> {cleaned[:80]}")

sub_df = pd.DataFrame(results)
print(f"\nSubmission shape: {sub_df.shape}")
sub_df

In [ ]:
sub_df.to_csv("submission.csv", index=False)
print("Submission saved!")

## 11. (Optional) Validation Evaluation

Uncomment to evaluate on validation set.

In [ ]:
# # Load val data
# val_df = pd.read_csv(os.path.join(CFG["competition_data"], "train.csv"))
# val_df = val_df.dropna(subset=["transliteration", "translation"])
# 
# predictions = []
# references = val_df["translation"].tolist()
# 
# with torch.inference_mode():
#     for idx, row in val_df.iterrows():
#         raw_text = str(row["transliteration"])
#         short = get_short_translation(raw_text)
#         if short is not None:
#             predictions.append(short)
#             continue
#         if is_broken_text(raw_text):
#             predictions.append("...")
#             continue
#         input_text = preprocess_input(raw_text)
#         inputs = tokenizer(input_text, max_length=496, padding=True, truncation=True, return_tensors="pt").to(DEVICE)
#         outputs = model.generate(input_ids=inputs.input_ids, attention_mask=inputs.attention_mask,
#                                  num_beams=12, max_new_tokens=496, length_penalty=1.3, early_stopping=True)
#         decoded = tokenizer.decode(outputs[0], skip_special_tokens=True)
#         predictions.append(postprocess_output(decoded, raw_text))
#         if (idx + 1) % 50 == 0:
#             print(f"[{idx+1}/{len(val_df)}]")
# 
# import sacrebleu
# bleu = sacrebleu.corpus_bleu(predictions, [references]).score
# chrf = sacrebleu.corpus_chrf(predictions, [references], word_order=2).score
# geo = math.sqrt(bleu * chrf)
# print(f"BLEU: {bleu:.2f}, chrF++: {chrf:.2f}, GeoMean: {geo:.2f}")